# Seasonality & Multi-Currency Calendar Demo

This notebook demonstrates the **seasonality extension** for liquidity forecasting with inline Matplotlib & Plotly visualizations.

### Visualizations Included:
- **STL Decomposition Plot** (Trend, Seasonal, and Residual components)
- **Seasonality Feature Importance Chart**

In [ ]:
import logging, sys, time
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from liquidity_forecast import LiquidityForecaster, PipelineConfig, synthetic
from liquidity_forecast.config import ModelConfig, SplitConfig
from liquidity_forecast.seasonality import (SeasonalityConfig, build_seasonal_features,
                                            CurrencyCalendar, account_open_mask, stl_decompose)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", stream=sys.stdout)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Calendar Setup & Pipeline Training

In [ ]:
t0 = time.time()
tables = synthetic.generate(start="2025-01-01", end="2025-09-01")
external = tables.pop("external"); tables.pop("stress_window")

sea_cfg = SeasonalityConfig(
    extra_holidays={"USD": ["2025-01-09"]},
    custom_events={"rmp_end": ["2025-02-11", "2025-03-25", "2025-05-06", "2025-06-17", "2025-07-29"],
                   "dividend_run": ["2025-03-28", "2025-06-27"]},
    event_currencies={"us_tax": ["USD"], "uk_tax": ["GBP"], "ust_coupon": ["USD"],
                      "rmp_end": ["EUR"]},
)
cal = CurrencyCalendar(["USD", "EUR", "GBP"], range(2024, 2027), sea_cfg)

cfg = PipelineConfig(
    model=ModelConfig(horizons=[1, 4], quantiles=[0.05, 0.5, 0.95], cv_folds=2,
                      lgb_params=dict(n_estimators=120, learning_rate=0.05, num_leaves=15, verbose=-1),
                      use_xgboost=False),
    split=SplitConfig(train_end="2025-06-30", valid_end="2025-07-31"),
)
fc = LiquidityForecaster(cfg, daily_horizons=[1],
                         extra_features=build_seasonal_features(sea_cfg),
                         open_mask=lambda panel: account_open_mask(panel, cal))
fc.fit(tables, external, run_cv=False)

## 2. STL Decomposition Plot

In [ ]:
d = stl_decompose(fc.panels["daily"], "CB_EUR_ECB")

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(d.index, d["trend"], label="Trend", color="navy")
axes[0].set_title("STL Decomposition - CB_EUR_ECB (Daily)")
axes[0].legend(loc="upper left")

axes[1].plot(d.index, d["seasonal"], label="Seasonal Component", color="green")
axes[1].legend(loc="upper left")

axes[2].plot(d.index, d["resid"], label="Residuals", color="crimson")
axes[2].legend(loc="upper left")
plt.tight_layout()
plt.show()

## 3. Seasonality Feature Importance Chart

In [ ]:
fi = fc.feature_importance("intraday", 4, top=15).reset_index()
fi.columns = ["feature", "importance"]

fig = px.bar(fi.sort_values("importance", ascending=True), x="importance", y="feature", orientation="h",
             title="Top 15 Features (Intraday Horizon 4h incl. Seasonality)", template="plotly_white", height=450)
fig.show()

### Data Analysis Summary

### Data Analysis Key Findings
- **STL Diagnostic Plot**: Clear separation between macro liquidity trends and recurring daily settlement cycles.
- **Seasonality Drivers**: Custom tax and central bank maintenance period markers show strong predictive importance.

### Insights or Next Steps
- Analyze seasonality components across distinct currency pairs.